Cell 1: Import thư viện & Cấu hình đường dẫn cây thư mục

In [8]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
DATA_DIR = os.path.join("..", "..", "data", "processed")
MODEL_DIR = os.path.join("..", "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)
books_path = os.path.join(DATA_DIR, "books_cleaned.csv")
if not os.path.exists(books_path):
    books_path = "books_cleaned.csv"
print(f"📁 Đường dẫn dữ liệu sách: {books_path}")
df_books = pd.read_csv(books_path, low_memory=False).dropna(how='all')
print(f"📚 Tổng số sách nạp vào: {len(df_books)}")

📁 Đường dẫn dữ liệu sách: ..\..\data\processed\books_cleaned.csv
📚 Tổng số sách nạp vào: 271360


Cell 2: Phân chia tập dữ liệu (Train 60% / Validation 20% / Test 20%)

In [9]:
train_df, temp_df = train_test_split(df_books, test_size=0.40, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
print(f"📊 Tập Train (60%): {len(train_df)} dòng")
print(f"📊 Tập Validation (20%): {len(val_df)} dòng")
print(f"📊 Tập Test (20%): {len(test_df)} dòng")

📊 Tập Train (60%): 162816 dòng
📊 Tập Validation (20%): 54272 dòng
📊 Tập Test (20%): 54272 dòng


Cell 3: Trích xuất đặc trưng văn bản & Huấn luyện TF-IDF

In [10]:
id_col = 'book_id' if 'book_id' in df_books.columns else ('ISBN' if 'ISBN' in df_books.columns else df_books.columns[0])
title_col = 'title' if 'title' in df_books.columns else ('Book-Title' if 'Book-Title' in df_books.columns else df_books.columns[1])
author_col = 'author' if 'author' in df_books.columns else ('Book-Author' if 'Book-Author' in df_books.columns else df_books.columns[2])
genre_col = 'genre' if 'genre' in df_books.columns else ('Publisher' if 'Publisher' in df_books.columns else None)

df_books[title_col] = df_books[title_col].fillna('')
df_books[author_col] = df_books[author_col].fillna('')
if genre_col and genre_col in df_books.columns:
    df_books[genre_col] = df_books[genre_col].fillna('')
    df_books['features'] = df_books[author_col] + " " + df_books[genre_col]
else:
    df_books['features'] = df_books[author_col]

print("⏳ Đang tính toán ma trận TF-IDF...")
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(df_books['features'])
indices = pd.Series(df_books.index, index=df_books[id_col]).drop_duplicates()
print(f"✅ Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")

⏳ Đang tính toán ma trận TF-IDF...
✅ Kích thước ma trận TF-IDF: (271360, 10000)


Cell 4: Hàm gợi ý mô hình & Chạy thử nghiệm (Recommendation Logic)

In [11]:
def recommend_similar_books(book_id, top_k=5):
    if book_id not in indices:
        return {"error": f"Không tìm thấy book_id: {book_id}"}
    idx = indices[book_id]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    target_vector = tfidf_matrix[idx]
    cosine_sim_vector = linear_kernel(target_vector, tfidf_matrix).flatten()
    sim_scores = sorted(list(enumerate(cosine_sim_vector)), key=lambda x: x[1], reverse=True)[1:top_k+1]

    similar_books = [{
        "book_id": str(df_books.iloc[i][id_col]),
        "title": str(df_books.iloc[i][title_col]),
        "similarity_score": round(float(score), 2)
    } for i, score in sim_scores]

    return {"target_book_id": str(book_id), "similar_books": similar_books}

sample_id = df_books[id_col].iloc[0]
print(f"🔍 [TEST] Kết quả gợi ý cho cuốn '{sample_id}':")
print(recommend_similar_books(sample_id, top_k=3))

🔍 [TEST] Kết quả gợi ý cho cuốn '0195153448':
{'target_book_id': '0195153448', 'similar_books': [{'book_id': '0195061101', 'title': 'Colonial Latin America', 'similarity_score': 1.0}, {'book_id': '0195101685', 'title': 'One Nation, Underprivileged: Why American Poverty Affects Us All', 'similarity_score': 0.9}, {'book_id': '0192752235', 'title': 'The Ennead', 'similarity_score': 0.83}]}


Cell 5: Đóng gói và lưu Model vào thư mục

In [12]:
export_path = os.path.join(MODEL_DIR, "content_based_model.pkl")
model_data_cb = {
    'tfidf_vectorizer': tfidf,
    'tfidf_matrix': tfidf_matrix,
    'df_books': df_books[[id_col, title_col, author_col]],
    'indices': indices,
    'train_ids': train_df[id_col].values,
    'val_ids': val_df[id_col].values,
    'test_ids': test_df[id_col].values,
    'id_col': id_col,
    'title_col': title_col
}

joblib.dump(model_data_cb, export_path, compress=3)
print(f"🎉 THÀNH CÔNG! Đã xuất file model chuẩn tại: {export_path}")

🎉 THÀNH CÔNG! Đã xuất file model chuẩn tại: ..\saved_models\content_based_model.pkl
